## Training and validation

**Goal:** Compare simple baselines and 2 pre-declared model pipelines on the 2020 validation period, then freeze one model family

In [6]:
#!pip install pandas numpy matplotlib missingno
#%pip install scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import missingno as msno
import joblib
import os
from pathlib import Path

#import sklearn.preprocessing

pd.options.display.max_rows=1000

import gc
import sys
import json
from IPython.display import display
import time

def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the parent directory containing src/data.py."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "data.py").is_file():
            return candidate

    raise FileNotFoundError("Could not find src/data.py")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
    


#from data import load_data, clean_prices, clean_stock_list, clean_financials
from config import ProjectConfig, ProjectPaths
from modeling import (
    build_model_pipeline
    ,chronological_split
    ,make_xy
    ,save_json
    ,select_recent_dates
    ,evaluate_predictions
    ,walk_forward_split
    ,make_fold_metric_record
)
from features import (
    infer_feature_columns
)


config = ProjectConfig()
paths = ProjectPaths(PROJECT_ROOT)
paths.ensure_output_dirs

print("Project root:", PROJECT_ROOT)
print("Imported load_data successfully")

print("Imported libraries sucessfully! yay")

Project root: c:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\Project_1_SupervisedLearning
Imported load_data successfully
Imported libraries sucessfully! yay


In [8]:
FAST_MODE = False
MODEL_NAMES = ['ridge', 'hist_gradient_boosting']

In [9]:

features_table = pd.read_parquet(paths.processed / "features_table.parquet")
numeric_features, categorical_features = infer_feature_columns(features_table)
splits, split_report = chronological_split(features_table, config)
train = splits['train']
valid = splits['validation']

model_train = select_recent_dates(train, max_dates=config.fast_train_dates if FAST_MODE else None)

print(f"training rows used: {len(model_train):,} of {len(train)}")
print(f"training dates used: {model_train['Date'].nunique()}")

training rows used: 1,395,521 of 1395521
training dates used: 731


## Baseline

In [6]:
evaluations = {}
prediction_outputs = {}
ic_outputs = {}
spread_outputs = {}

baseline_predictions = {
    "baseline_zero": np.zeros(len(valid)),
    "baseline_momentum20": valid["Momentum20Days"].fillna(0.0).to_numpy(),
}

for name, predictions in baseline_predictions.items():
    metrics, ranked, daily_ic, daily_spread_return = evaluate_predictions(
        valid,
        predictions,
        config,
        strict_portfolio_size=True,
    )
    evaluations[name] = metrics
    prediction_outputs[name] = ranked
    ic_outputs[name] = daily_ic
    spread_outputs[name] = daily_spread_return

display(pd.DataFrame(evaluations).T)

c:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\.venv\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
c:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\.venv\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
c:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\.venv\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
c:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\.venv\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient 

,MAE,RMSE,R2,MeanDailyIC,MedianDailyIC,MeanDailySpreadReturn,MedianDailySpreadReturn,DailySpreadReturnStd,SpreadSharpeRatio
baseline_zero,0.019717,0.000842,-0.000112,NaN,NaN,0.006302,0.003330,0.539116,0.011690
baseline_momentum20,0.095870,0.020363,-23.173361,0.004797,0.004428,0.040366,0.118331,3.557256,0.011348


## Fit candidate pipelines

- **Ridge:** a stable linear benchmark with median imputation, missingness
  indicators, scaling, and one-hot categories.
- **Histogram gradient boosting:** a nonlinear model with median imputation
  and unknown-safe ordinal category encoding.

#### Walk-forward validation

Ordinary shuffled K-fold cross validation is inappropriate because it could train on future dates and validate on past dates. Instead, expanding window validation is used here:

- Fold 1: train on 2017 and validate on 2018
- Fold 2: train on 2017-2018 and validate on 2019
- Fold 3: train on 2017-2019 and validate on 2020

The last training dates are removed in every fold because each Target depends on the following two trading dates. This is to prevent data leakage. 
The 2021 test data is excluded from all model selection and is opened only in notebook 05

In [10]:
WALK_FORWARD_FOLDS = [
    {
        "name": "validate_2018"
        ,"train_end": "2017-12-31"
        ,"valid_start": "2018-01-01"
        ,"valid_end": "2018-12-31"
    }
    ,{
        "name": "validate_2019"
        ,"train_end": "2018-12-31"
        ,"valid_start": "2019-01-01"
        ,"valid_end": "2019-12-31"
    }
    ,{
        "name": "validate_2020"
        ,"train_end": "2019-12-31"
        ,"valid_start": "2020-01-01"
        ,"valid_end": "2020-12-31"
    }
]

for fold in WALK_FORWARD_FOLDS:
    assert pd.Timestamp(fold["train_end"]) < pd.Timestamp(fold["valid_start"])
    assert pd.Timestamp(fold['valid_end']) < pd.Timestamp(config.test_start)

In [20]:

X_train, y_train = make_xy(model_train, numeric_features, categorical_features)
X_valid, y_valid = make_xy(valid, numeric_features, categorical_features)

fitted_models = {}
training_seconds = {}
fold_metric_records = []
#fold_audit_records = []
daily_ic_record = {}
daily_spread_record = {}

ranked_2020_outputs = {}

momentum_name = "baseline_momentum20"



for fold in WALK_FORWARD_FOLDS:
    fold_name = fold['name']
    print(f"Training fold {fold_name}")
    full_train = features_table[features_table['Date'] <= fold['train_end']]
    fold_train, fold_valid, train_purged_dates = walk_forward_split(
            features_table = features_table
            ,train_end = fold['train_end']
            ,valid_start = fold['valid_start']
            ,valid_end = fold['valid_end']
            ,purged_dates = config.purge_dates
    )
    
    X_fold_train, y_fold_train = make_xy(fold_train, numeric_features, categorical_features)
    X_fold_valid, y_fold_valid = make_xy(fold_valid, numeric_features, categorical_features)
    
    assert len(X_fold_train) > 0, f"Fold {fold_name} has no training data"
    assert len(X_fold_valid) > 0, f"Fold {fold_name} has no validation data"
    assert len(X_fold_train) == len(y_fold_train), f"Fold {fold_name} has mismatched training data and labels"
    assert len(X_fold_valid) == len(y_fold_valid), f"Fold {fold_name} has mismatched validation data and labels"
    
    ''' Baseline model
    Predict momentum for all stocks
    
    At date t:
        Momentum20days is available as it is calculated from the past 20 days of returns. 
        Target is not yet available 
        
        Past prices ─────────── Date t ───── t+1 ───── t+2
                │                    │                      │
                └─ Momentum20Days ───┘                      │
                                    └──── Target ───────────┘
    '''
    momentum_predictions = fold_valid["Momentum20Days"].fillna(0.0).to_numpy()
    momentum_metrics, momentum_ranked, momentum_daily_ic, momentum_daily_spread_return = evaluate_predictions(
            df = fold_valid
            ,predictions = momentum_predictions
            ,config = config
            ,strict_portfolio_size = True
    )
    
    momentum_key = (momentum_name, fold_name)
    
    fold_metric_records.append(
        make_fold_metric_record(
            train_full = full_train
            ,train_purged = fold_train
            ,valid = fold_valid
            ,train_purged_dates = train_purged_dates
            ,model_name = momentum_name
            ,fold_name = fold_name
            ,metrics = momentum_metrics
            ,training_seconds = 0.0
        )
    )
    
    daily_ic_record[momentum_key] = momentum_daily_ic
    daily_spread_record[momentum_key] = momentum_daily_spread_return
    
    if fold_name == 'validate_2020':
        ranked_2020_outputs[momentum_key] = momentum_ranked
        
    
    '''
    Supervised model pipelines
    '''
    for model_name in MODEL_NAMES:
        print(f"\n--------------------------Training {model_name} on fold {fold_name}...------------------------------")
        '''
        Create a completely new pipeline for every fold to avoid data leakage.
        Its median, scaling and category encodings are therefore fitted only on the training data of that fold.
        '''
        pipeline = build_model_pipeline(
            model_name = model_name
            ,numeric_features = numeric_features
            ,categorical_features = categorical_features
            ,random_seed = config.random_seed
            ,fast_mode = FAST_MODE
        )
        output_key = (model_name, fold_name)
        started = time.perf_counter()
        pipeline.fit(X_fold_train, y_fold_train)
        fit_seconds = time.perf_counter() - started
        training_seconds[output_key] = fit_seconds
        
        valid_predictions = np.asarray(pipeline.predict(X_fold_valid), dtype=float).reshape(-1)
        
        if len(valid_predictions) != len(y_fold_valid):
            raise ValueError(f"Fold {fold_name} model {model_name} produced {len(valid_predictions)} predictions, but validation set has {len(y_fold_valid)} samples.")
        
        if not np.isfinite(valid_predictions).all(): #
            raise ValueError(f"Fold {fold_name} model {model_name} produced non-finite predictions.")
        
        metrics, ranked, daily_ic, daily_spread_return = evaluate_predictions(
            df = fold_valid
            ,predictions = valid_predictions
            ,config = config
            ,strict_portfolio_size = True
        )
        
        fold_metric_records.append(
        make_fold_metric_record(
            train_full = full_train
            ,train_purged = fold_train
            ,valid = fold_valid
            ,train_purged_dates = train_purged_dates
            ,model_name = model_name
            ,fold_name = fold_name
            ,metrics = metrics
            ,training_seconds = fit_seconds
            )
        )
        
        daily_ic_record[output_key] = daily_ic
        daily_spread_record[output_key] = daily_spread_return
        
        if fold_name == 'validate_2020':
            ranked_2020_outputs[output_key] = ranked
        
        print(f"\nFinished {model_name} on fold {fold_name} in {training_seconds[output_key]:.1f} seconds")
        
        '''
        Fold models are temporary. Notebool 05 will refit the selected on all development data (train + validation) and save the final model to disk.
        '''
        del pipeline
        del valid_predictions
        del ranked
        del daily_ic
        del daily_spread_return
        
        gc.collect()
        
    if fold_name == 'validate_2020':
        print(f"Finished all models on fold {fold_name}.")
        validation_2020 = fold_valid.copy()
        
    del X_fold_train
    del y_fold_train
    del X_fold_valid
    del y_fold_valid
    del fold_train
    del fold_valid
    
    gc.collect() #gabage collection to free up memory after each fold
        
            


Training fold validate_2018

--------------------------Training ridge on fold validate_2018...------------------------------

--------------------------Finished ridge on fold validate_2018 in 5.7 seconds------------------------------

--------------------------Training hist_gradient_boosting on fold validate_2018...------------------------------

--------------------------Finished hist_gradient_boosting on fold validate_2018 in 7.5 seconds------------------------------
Training fold validate_2019

--------------------------Training ridge on fold validate_2019...------------------------------

--------------------------Finished ridge on fold validate_2019 in 9.9 seconds------------------------------

--------------------------Training hist_gradient_boosting on fold validate_2019...------------------------------

--------------------------Finished hist_gradient_boosting on fold validate_2019 in 21.7 seconds------------------------------
Training fold validate_2020


c:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\.venv\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]



--------------------------Training ridge on fold validate_2020...------------------------------


c:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\.venv\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]



--------------------------Finished ridge on fold validate_2020 in 16.5 seconds------------------------------

--------------------------Training hist_gradient_boosting on fold validate_2020...------------------------------


c:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\.venv\Lib\site-packages\pandas\core\nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]



--------------------------Finished hist_gradient_boosting on fold validate_2020 in 31.0 seconds------------------------------
Finished all models on fold validate_2020.


In [21]:
fold_results = pd.DataFrame(fold_metric_records).sort_values(by=['fold_name', 'model_name']).reset_index(drop=True)
metric_columns = [
    'MAE'
    ,'RMSE'
    ,'R2'
    ,'MeanDailyIC'
    ,'MedianDailyIC'
    ,'MeanDailySpreadReturn'
    ,'MedianDailySpreadReturn'
    ,'DailySpreadReturnStd'
    ,'SpreadSharpeRatio'
]

display(fold_results[['model_name', 'fold_name', *metric_columns, 'training_seconds']].set_index(['model_name', 'fold_name']))

,,MAE,RMSE,R2,MeanDailyIC,MedianDailyIC,MeanDailySpreadReturn,MedianDailySpreadReturn,DailySpreadReturnStd,SpreadSharpeRatio,training_seconds
model_name,fold_name,,,,,,,,,,
baseline_momentum20,validate_2018,0.076274,0.015854,-25.791748,-0.006223,-0.003407,-0.079223,0.056908,1.668201,-0.047490,0.000000
hist_gradient_boosting,validate_2018,0.016473,0.000603,-0.018433,0.003532,0.003514,0.254655,0.234974,1.531813,0.166244,7.470883
ridge,validate_2018,0.016459,0.000603,-0.019057,-0.011625,-0.007369,0.121545,0.199851,2.017658,0.060240,5.661548
baseline_momentum20,validate_2019,0.071586,0.012367,-26.540884,-0.015597,-0.015323,-0.146070,-0.097125,1.455155,-0.100381,0.000000
hist_gradient_boosting,validate_2019,0.014461,0.000465,-0.036429,0.010785,0.012898,0.282243,0.222803,1.327823,0.212560,21.741276
ridge,validate_2019,0.014313,0.000454,-0.011346,0.004141,0.014534,0.119522,0.240144,1.556636,0.076782,9.928285
baseline_momentum20,validate_2020,0.095608,0.020262,-23.138724,0.004570,0.002727,0.033080,0.076718,3.543513,0.009335,0.000000
hist_gradient_boosting,validate_2020,0.020045,0.000869,-0.035854,0.016209,0.017751,0.432224,0.516698,2.108342,0.205006,31.044963
ridge,validate_2020,0.019661,0.000835,0.005783,0.005733,0.011361,0.392435,0.349397,3.515401,0.111633,16.528691


In [25]:
model_fold_counts = fold_results.loc[fold_results['model_name'].isin(MODEL_NAMES)].groupby('model_name')['fold_name'].nunique()

assert model_fold_counts.eq(len(WALK_FORWARD_FOLDS)).all(), f"Not all models were trained on all folds. Model fold counts: {model_fold_counts.to_dict()}"

display(model_fold_counts.to_frame('name=countfolds'))

,name=countfolds
model_name,
hist_gradient_boosting,3
ridge,3


In [27]:
holdout_2020 = fold_results.loc[
    fold_results['fold_name'].eq('validate_2020'), ['model_name', *metric_columns, 'training_seconds']].sort_values('SpreadSharpeRatio', ascending=False)
display(holdout_2020)

,model_name,MAE,RMSE,R2,MeanDailyIC,MedianDailyIC,MeanDailySpreadReturn,MedianDailySpreadReturn,DailySpreadReturnStd,SpreadSharpeRatio,training_seconds
7,hist_gradient_boosting,0.020045,0.000869,-0.035854,0.016209,0.017751,0.432224,0.516698,2.108342,0.205006,31.044963
8,ridge,0.019661,0.000835,0.005783,0.005733,0.011361,0.392435,0.349397,3.515401,0.111633,16.528691
6,baseline_momentum20,0.095608,0.020262,-23.138724,0.004570,0.002727,0.033080,0.076718,3.543513,0.009335,0.000000


## Compare

In [30]:
cv_summary = fold_results.groupby('model_name')[metric_columns].agg(['mean','std'])
cv_summary.columns = [f"{metric}_{statistic}" for metric, statistic in cv_summary.columns]
cv_summary.insert(0, 'NumberOfFolds',fold_results.groupby('model_name')['fold_name'].nunique())
cv_summary = cv_summary.sort_values('SpreadSharpeRatio_mean', ascending=False)

In [31]:
spread_by_year = fold_results.pivot(index='model_name',columns='fold_name',values='SpreadSharpeRatio')
ic_by_year = fold_results.pivot(index='model_name',columns='fold_name', values='SpreadSharpeRatio')

display(spread_by_year)
display(ic_by_year)

fold_name,validate_2018,validate_2019,validate_2020
model_name,,,
baseline_momentum20,-0.047490,-0.100381,0.009335
hist_gradient_boosting,0.166244,0.212560,0.205006
ridge,0.060240,0.076782,0.111633


fold_name,validate_2018,validate_2019,validate_2020
model_name,,,
baseline_momentum20,-0.047490,-0.100381,0.009335
hist_gradient_boosting,0.166244,0.212560,0.205006
ridge,0.060240,0.076782,0.111633


In [32]:
comparison_columns = [
    "SpreadSharpeRatio_mean",
    "SpreadSharpeRatio_std",
    "MeanDailyIC_mean",
    "MeanDailyIC_std",
    "RMSE_mean",
    "RMSE_std",
]

display(cv_summary[comparison_columns])


,SpreadSharpeRatio_mean,SpreadSharpeRatio_std,MeanDailyIC_mean,MeanDailyIC_std,RMSE_mean,RMSE_std
model_name,,,,,,
hist_gradient_boosting,0.194604,0.024849,0.010175,0.006360,0.000646,0.000205
ridge,0.082885,0.026234,-0.000584,0.009595,0.000631,0.000192
baseline_momentum20,-0.046179,0.054870,-0.005750,0.010092,0.016161,0.003956


In [33]:
supervised_summary = cv_summary.loc[cv_summary.index.isin(MODEL_NAMES)].copy()

selected_model = supervised_summary["SpreadSharpeRatio_mean"].idxmax()


print("Selected supervised model:", selected_model)

display(
    supervised_summary.loc[
        [selected_model],
        comparison_columns,
    ]
)

Selected supervised model: hist_gradient_boosting


,SpreadSharpeRatio_mean,SpreadSharpeRatio_std,MeanDailyIC_mean,MeanDailyIC_std,RMSE_mean,RMSE_std
model_name,,,,,,
hist_gradient_boosting,0.194604,0.024849,0.010175,0.00636,0.000646,0.000205


In [34]:
fold_results.to_csv(
    paths.reports / "walk_forward_fold_metrics.csv",
    index=False,
)

cv_summary.to_csv(
    paths.reports / "walk_forward_summary.csv",
    index_label="model",
)


In [43]:
selected_row = supervised_summary.loc[selected_model]

selection = {
    "selected_model": selected_model
    ,"fast_mode": FAST_MODE
    ,"selection_metric": "Mean SpreadSharpeRatio across expanding-window folds (3 folds in total)"
    ,"walk_forward_folds": [
        {
            "name": fold["name"]
            ,"train_end": fold["train_end"]
            ,"valid_start": fold["valid_start"]
            ,"valid_end": fold["valid_end"]
        }
        for fold in WALK_FORWARD_FOLDS
    ]
    ,"mean_spread_sharpe": float(selected_row["SpreadSharpeRatio_mean"])
    ,"std_jpx_spread_sharpe": float(selected_row["SpreadSharpeRatio_std"])
    ,"mean_daily_spearman_ic": float(selected_row["MeanDailyIC_mean"])
    ,"std_daily_spearman_ic": float(selected_row["MeanDailyIC_std"])
    ,"note": (
        "Model family was selected without examining "
        "2021 test performance."
    )
}

save_json(selection, paths.reports / "selected_model.json")